In [2]:
import run as run
import chephy_model as cpm
import numpy as np
import torch
from torch_geometric.data import HeteroData
from torch_geometric.transforms import ToUndirected
import pandas as pd
from sklearn.model_selection import train_test_split


/home/dsi/chenbis/anaconda3/envs/chephy2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def get_epitope_features(epitopes, atchley_dict):
    # Create epitope features (mean of Atchley vectors per sequence)
    epitope_features = []
    for epitope in epitopes:
        vecs = [list(atchley_dict[aa].values()) for aa in epitope if aa in atchley_dict]
        if vecs:
            mean_vec = np.mean(vecs, axis=0)
        else:
            mean_vec = np.zeros(len(next(iter(atchley_dict.values()))))
        epitope_features.append(mean_vec)

    epitope_features = torch.tensor(epitope_features, dtype=torch.float)
    return epitope_features


def get_tcr_features(sequences_list, atchley_dict):
    # Use Atchley vectors as node features for TCRs
    tcr_features = []
    for seq in sequences_list:
        vecs = [list(atchley_dict[aa].values()) for aa in seq if aa in atchley_dict]
        tcr_features.append(np.concatenate(vecs))
    tcr_features = torch.tensor(tcr_features, dtype=torch.float)  # [num_tcr, 8*5]
    return tcr_features

In [4]:
def build_hetero_graph(data, chephy_matrix, sequences_list):
    # Load Atchley dictionary
    atchley_df = pd.read_csv("/home/dsi/chenbis/repos/sol_lab/files/atchley.csv")
    atchley_dict = atchley_df.set_index('amino.acid').to_dict(orient='index')    
    epitopes = list(set(data["antigen.epitope"]))

    epitope_features = get_epitope_features(epitopes, atchley_dict)
    tcr_features = get_tcr_features(sequences_list, atchley_dict)

    # Get edges from the data
    edge_index = torch.tensor(data[['local_tcr_index', 'epitope_index']].dropna().astype(int).values.T, dtype=torch.long)  # shape [2, num_edges]
    
    # Convert chephy_matrix (distance) into edge index for TCR similarity
    threshold = 1  # You can tune this value

    rows, cols = np.where((chephy_matrix < threshold) & (chephy_matrix > 0))  # Exclude self-loops
    edge_pairs = list(zip(rows, cols))    
    
    tcr_sim_edge_index = torch.tensor(edge_pairs, dtype=torch.long).T  # shape [2, num_edges]

    # Similarity = 1 - distance
    edge_weights = 1 - chephy_matrix[rows, cols]
    edge_weights = torch.tensor(edge_weights, dtype=torch.float)
    edge_scores = torch.tensor(data['vdjdb.score'].values, dtype=torch.float)
        
    data_hetero = HeteroData()

    data_hetero['tcr'].x = tcr_features
    data_hetero['epitope'].x = epitope_features
    data_hetero['tcr', 'binds', 'epitope'].edge_index = edge_index
    data_hetero['tcr', 'binds', 'epitope'].edge_attr = edge_scores
    data_hetero['epitope', 'rev_binds', 'tcr'].edge_index = edge_index
    data_hetero['epitope', 'rev_binds', 'tcr'].edge_attr = edge_scores
    data_hetero['tcr', 'similar', 'tcr'].edge_index = tcr_sim_edge_index
    data_hetero['tcr', 'similar', 'tcr'].edge_attr = edge_weights


    num_tcr = len(sequences_list)
    num_epitopes = len(epitopes)
    # [num_tcr, num_epitopes], multi-label format
    tcr_labels = torch.zeros((num_tcr, num_epitopes), dtype=torch.float)
    train_mask = torch.ones(num_tcr, dtype=torch.bool)
    data_hetero['tcr'].train_mask = train_mask


    for row in data.itertuples():
        tcr_idx = row.local_tcr_index
        epi_idx = row.epitope_index
        if tcr_idx != -1 and epi_idx != -1:
            tcr_labels[tcr_idx, epi_idx] = 1.0

    data_hetero['tcr'].y = tcr_labels  # shape: [num_tcr, num_epitopes]


    # # create masks for training, validation and testing
    # valid_indices = (tcr_labels.sum(dim=1) > 0).nonzero(as_tuple=True)[0]
    # train_idx, temp_idx = train_test_split(valid_indices, test_size=0.3, random_state=42)
    # val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

    # num_tcr = data_hetero['tcr'].num_nodes
    # train_mask = torch.zeros(num_tcr, dtype=torch.bool)
    # val_mask = torch.zeros(num_tcr, dtype=torch.bool)
    # test_mask = torch.zeros(num_tcr, dtype=torch.bool)

    # train_mask[train_idx] = True
    # val_mask[val_idx] = True
    # test_mask[test_idx] = True
    # data_hetero['tcr'].train_mask = train_mask
    # data_hetero['tcr'].val_mask = val_mask
    # data_hetero['tcr'].test_mask = test_mask

    return data_hetero



In [5]:
from sklearn.model_selection import train_test_split
import numpy as np

def split_tcrs_with_epitope_coverage(data, test_size=0.2, random_state=42):
    # Mapping: epitope -> tcr indices
    epitope_to_tcrs = data.groupby('epitope_index')['tcr_index'].apply(list).to_dict()


    # 1. Ensure at least one TCR per epitope in training
    mandatory_train_tcrs = set()
    for tcrs in epitope_to_tcrs.values():
        mandatory_train_tcrs.add(tcrs[0])  # Pick first TCR

    all_tcrs = set(data['tcr_index'].unique())
    remaining_tcrs = np.array(list(all_tcrs - mandatory_train_tcrs))

    # 2. Randomly split the remaining TCRs
    if len(remaining_tcrs) > 0:
        train_rest, test = train_test_split(
            remaining_tcrs, test_size=test_size, random_state=random_state
        )
    else:
        train_rest, test = [], []

    # 3. Merge mandatory TCRs with the rest
    train = np.array(list(mandatory_train_tcrs)) if isinstance(mandatory_train_tcrs, set) else mandatory_train_tcrs
    train = np.concatenate([train, train_rest])

    return train, test


In [6]:
input_file = "../files/vdjdb_score3.csv"
max_dist=1
label_header = "antigen.epitope"
cdr3_header = "cdr3"

data = run.load_dataframe(input_file)

# data = read_csv_files_from_folder()

data = run.prepare_data(data, cdr3_header, label_header)    
data = cpm.truncate_sequences(data, cdr3_header, 4, 4)

sequences = set(data["cdr3_truncated"])
epitopes = list(set(data["antigen.epitope"]))

chephy_matrix, hamming_matrix, sequences_list = cpm.find_che_phy_dist(sequences)

data['tcr_index'] = data['cdr3_truncated'].apply(lambda x: sequences_list.index(x) if x in sequences_list else -1)
data['epitope_index'] = data['antigen.epitope'].apply(lambda x: epitopes.index(x) if x in epitopes else -1)

sequence_indices = np.arange(len(sequences_list))


In [7]:
# Split into train and test (80% train, 20% test)
train_indices, test_indices = split_tcrs_with_epitope_coverage(data)

# Extract submatrix for training set
train_distance_matrix = chephy_matrix[np.ix_(train_indices, train_indices)]

train_data = data[data['tcr_index'].isin(train_indices)].copy()
train_sequences_list = [sequences_list[i] for i in train_indices]

index_mapping = {orig_idx: local_idx for local_idx, orig_idx in enumerate(train_indices)}

# Add the local train index to train_data
train_data['local_tcr_index'] = train_data['tcr_index'].map(index_mapping)


# Create a mapping of original epitope indices to local indices in train_data
train_epitopes = sorted(list(set(train_data['epitope_index'])))
epitope_index_mapping = {orig_idx: local_idx for local_idx, orig_idx in enumerate(train_epitopes)}


In [8]:
data_hetero = build_hetero_graph(train_data, train_distance_matrix, train_sequences_list)
# add reverse edged from epitope to tcr


/tmp/ipykernel_3945309/312848608.py:12: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647327489/work/torch/csrc/utils/tensor_new.cpp:278.)
  epitope_features = torch.tensor(epitope_features, dtype=torch.float)


In [10]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import HGTConv, Linear
from sklearn.metrics import roc_auc_score, average_precision_score

class TCREpitopeGNN(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels, num_layers, metadata):
        super().__init__()
        self.lin_dict = torch.nn.ModuleDict()
        for node_type in metadata[0]:
            self.lin_dict[node_type] = Linear(-1, hidden_channels)

        self.convs = torch.nn.ModuleList()
        for _ in range(num_layers):
            conv = HGTConv(
                in_channels=hidden_channels,
                out_channels=hidden_channels,
                metadata=metadata,
                heads=2,
            )
            self.convs.append(conv)

        self.binding_predictor = torch.nn.Sequential(
            Linear(hidden_channels * 2, hidden_channels),
            torch.nn.ReLU(),
            Linear(hidden_channels, out_channels)
        )

    def forward(self, data):
        x_dict, edge_index_dict = data.x_dict, data.edge_index_dict

        x_dict = {node_type: self.lin_dict[node_type](x) for node_type, x in x_dict.items()}

        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)

        # Save embeddings
        return x_dict


In [11]:
def train(model, data, optimizer, criterion):
    model.train()
    optimizer.zero_grad()

    x_dict = model(data)

    # Get embeddings
    tcr_embeds = x_dict['tcr']
    epitope_embeds = x_dict['epitope']

    # Get binding edges
    bind_src, bind_dst = data['tcr', 'binds', 'epitope'].edge_index

    # Mask: use only training TCRs
    train_mask = data['tcr'].train_mask
    train_edge_mask = train_mask[bind_src]

    src_feats = tcr_embeds[bind_src[train_edge_mask]]
    dst_feats = epitope_embeds[bind_dst[train_edge_mask]]

    edge_input = torch.cat([src_feats, dst_feats], dim=-1)
    pred = model.binding_predictor(edge_input).squeeze()

    target = data['tcr', 'binds', 'epitope'].edge_attr[train_edge_mask]

    loss = criterion(pred, target)
    loss.backward()
    optimizer.step()

    return loss.item()


In [18]:
def build_test_hetero_graph(test_data, chephy_matrix, test_sequences_list, epitope_list, epitope_index_mapping):
    atchley_df = pd.read_csv("/home/dsi/chenbis/repos/sol_lab/files/atchley.csv")
    atchley_dict = atchley_df.set_index('amino.acid').to_dict(orient='index')

    epitope_features = get_epitope_features(epitope_list, atchley_dict)
    tcr_features = get_tcr_features(test_sequences_list, atchley_dict)

    edge_index = torch.tensor(test_data[['local_tcr_index', 'local_epitope_index']].dropna().astype(int).values.T, dtype=torch.long)
    edge_scores = torch.tensor(test_data['vdjdb.score'].values, dtype=torch.float)

    data_hetero = HeteroData()
    data_hetero['tcr'].x = tcr_features
    data_hetero['epitope'].x = epitope_features
    data_hetero['tcr', 'binds', 'epitope'].edge_index = edge_index
    data_hetero['tcr', 'binds', 'epitope'].edge_attr = edge_scores

    return data_hetero


In [13]:
def evaluate(model, data):
    model.eval()
    with torch.no_grad():
        x_dict = model(data)
        tcr_embeds = x_dict['tcr']
        epitope_embeds = x_dict['epitope']

        bind_src, bind_dst = data['tcr', 'binds', 'epitope'].edge_index

        src_feats = tcr_embeds[bind_src]
        dst_feats = epitope_embeds[bind_dst]

        edge_input = torch.cat([src_feats, dst_feats], dim=-1)
        pred = model.binding_predictor(edge_input).squeeze()

        target = data['tcr', 'binds', 'epitope'].edge_attr

        pred_probs = torch.sigmoid(pred).cpu().numpy()
        target = target.cpu().numpy()

        auc = roc_auc_score(target, pred_probs)
        ap = average_precision_score(target, pred_probs)

    return auc, ap


In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = TCREpitopeGNN(hidden_channels=64, out_channels=1, num_layers=2, metadata=data_hetero.metadata()).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
criterion = torch.nn.BCEWithLogitsLoss()

data_hetero = data_hetero.to(device)

for epoch in range(1, 201):
    loss = train(model, data_hetero, optimizer, criterion)
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {loss:.4f}')


Epoch 10, Loss: 0.0899
Epoch 20, Loss: -6.5416
Epoch 30, Loss: -128.0707
Epoch 40, Loss: -1644.9476
Epoch 50, Loss: -16361.7236
Epoch 60, Loss: -125034.1562
Epoch 70, Loss: -766190.8125
Epoch 80, Loss: -4026030.5000
Epoch 90, Loss: -17914152.0000
Epoch 100, Loss: -70344632.0000
Epoch 110, Loss: -174778672.0000
Epoch 120, Loss: -441794720.0000
Epoch 130, Loss: -1203463424.0000
Epoch 140, Loss: -3147801856.0000
Epoch 150, Loss: -7802880512.0000
Epoch 160, Loss: -17748797440.0000
Epoch 170, Loss: -39433703424.0000
Epoch 180, Loss: -45753769984.0000
Epoch 190, Loss: -56991535104.0000
Epoch 200, Loss: -20412004352.0000


In [19]:
# Build test_data, test_sequences_list, epitope_index_mapping
test_data = data[data['tcr_index'].isin(test_indices)].copy()
test_sequences_list = [sequences_list[i] for i in test_indices]

epitope_list = epitopes

index_mapping_test = {orig_idx: local_idx for local_idx, orig_idx in enumerate(test_indices)}
test_data['local_tcr_index'] = test_data['tcr_index'].map(index_mapping_test)
test_data['local_epitope_index'] = test_data['epitope_index'].map(epitope_index_mapping)

test_distance_matrix = chephy_matrix[np.ix_(test_indices, test_indices)]

data_hetero_test = build_test_hetero_graph(
    test_data,
    test_distance_matrix,
    test_sequences_list,
    epitope_list,
    epitope_index_mapping
)
data_hetero_test = data_hetero_test.to(device)

# Evaluate
auc, ap = evaluate(model, data_hetero_test)
print(f'Test ROC-AUC: {auc:.4f}, Test PR-AUC: {ap:.4f}')


Test ROC-AUC: nan, Test PR-AUC: 0.0000


/home/dsi/chenbis/anaconda3/envs/chephy2/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/dsi/chenbis/anaconda3/envs/chephy2/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
